# Task 35: Polynomial and RBF Kernel SVM

This notebook builds both kernels from their formulas, trains a kernelized hinge-loss classifier, and compares predictive metrics and support-vector counts by class.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

candidates = [Path("heart.csv"), Path("Task_35_Kernel_SVM_From_Scratch/heart.csv"), Path("Heart_Disease_Tasks/Task_35_Kernel_SVM_From_Scratch/heart.csv")]
csv_path = next(path for path in candidates if path.is_file())
data = pd.read_csv(csv_path).drop_duplicates().reset_index(drop=True)
feature_names = [name for name in ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"] if name in data.columns]
X = data[feature_names].to_numpy(dtype=float)
y = data["target"].to_numpy(dtype=int)

rng = np.random.default_rng(35)
train_parts, test_parts = [], []
for label in np.unique(y):
    indices = np.flatnonzero(y == label)
    rng.shuffle(indices)
    split_at = int(round(len(indices) * 0.8))
    train_parts.append(indices[:split_at])
    test_parts.append(indices[split_at:])
train_idx, test_idx = np.concatenate(train_parts), np.concatenate(test_parts)
rng.shuffle(train_idx)
rng.shuffle(test_idx)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
means, scales = X_train.mean(axis=0), X_train.std(axis=0)
scales[scales == 0] = 1.0
X_train, X_test = (X_train - means) / scales, (X_test - means) / scales


def manual_kernel(left, right, kind, degree=2, gamma=None):
    dot = left @ right.T
    if kind == "polynomial":
        return (dot / left.shape[1] + 1.0) ** degree
    if kind == "rbf":
        gamma = 1.0 / left.shape[1] if gamma is None else gamma
        left_norm = np.sum(left * left, axis=1)[:, None]
        right_norm = np.sum(right * right, axis=1)[None, :]
        squared_distance = np.maximum(left_norm + right_norm - 2 * dot, 0.0)
        return np.exp(-gamma * squared_distance)
    raise ValueError("kind must be polynomial or rbf")


def train_kernel_svm(kind, degree=2, gamma=None, C=1.0, max_passes=10):
    signed_y = np.where(y_train == 1, 1.0, -1.0)
    gram = manual_kernel(X_train, X_train, kind, degree, gamma)
    alpha = np.zeros(len(y_train))
    bias = 0.0
    errors = -signed_y.copy()
    unchanged = 0
    for _ in range(max_passes):
        updates = 0
        for i in range(len(y_train)):
            error_i, yi, ai = errors[i], signed_y[i], alpha[i]
            if not ((yi * error_i < -1e-3 and ai < C - 1e-8) or (yi * error_i > 1e-3 and ai > 1e-8)):
                continue
            gaps = np.abs(error_i - errors)
            gaps[i] = -1.0
            j = int(np.argmax(gaps))
            error_j, yj, old_i, old_j = errors[j], signed_y[j], alpha[i], alpha[j]
            if yi != yj:
                low, high = max(0.0, old_j - old_i), min(C, C + old_j - old_i)
            else:
                low, high = max(0.0, old_i + old_j - C), min(C, old_i + old_j)
            eta = 2 * gram[i, j] - gram[i, i] - gram[j, j]
            if high - low < 1e-10 or eta >= -1e-12:
                continue
            new_j = float(np.clip(old_j - yj * (error_i - error_j) / eta, low, high))
            if abs(new_j - old_j) < 1e-6:
                continue
            new_i = old_i + yi * yj * (old_j - new_j)
            alpha[i], alpha[j] = new_i, new_j
            bias_i = bias - error_i - yi * (new_i - old_i) * gram[i, i] - yj * (new_j - old_j) * gram[i, j]
            bias_j = bias - error_j - yi * (new_i - old_i) * gram[i, j] - yj * (new_j - old_j) * gram[j, j]
            if 1e-8 < new_i < C - 1e-8:
                new_bias = bias_i
            elif 1e-8 < new_j < C - 1e-8:
                new_bias = bias_j
            else:
                new_bias = (bias_i + bias_j) / 2
            errors += yi * (new_i - old_i) * gram[:, i]
            errors += yj * (new_j - old_j) * gram[:, j]
            errors += new_bias - bias
            bias = float(new_bias)
            updates += 1
        unchanged = unchanged + 1 if updates == 0 else 0
        if unchanged >= 3:
            break
    support = alpha > 1e-8
    scores = manual_kernel(X_test, X_train, kind, degree, gamma) @ (alpha * signed_y) + bias
    predictions = (scores >= 0).astype(int)
    accuracy = float(np.mean(predictions == y_test))
    counts = {label: int(np.sum(support & (y_train == label))) for label in (0, 1)}
    return accuracy, counts


results = []
for kernel_name, options in [("polynomial degree 2", {"kind": "polynomial"}), ("RBF", {"kind": "rbf", "gamma": 1 / X_train.shape[1]})]:
    accuracy, support_counts = train_kernel_svm(**options)
    results.append({"Kernel": kernel_name, "Accuracy": accuracy, "Support vectors: class 0": support_counts[0], "Support vectors: class 1": support_counts[1]})

print(f"Unique records used: {len(data)}")
pd.DataFrame(results).set_index("Kernel")

Unique records used: 302


,Accuracy,Support vectors: class 0,Support vectors: class 1
Kernel,,,
polynomial degree 2,0.786885,23,24
RBF,0.721311,18,18
